# 07 - Blocking and Candidate Generation

## Objective

Menguji beberapa strategi blocking untuk mengurangi jumlah pasangan yang perlu dibandingkan sebelum fuzzy matching. Notebook ini tidak melakukan merge atau menghapus baris.

## Research Questions

1. Seberapa besar pengurangan candidate pair dibandingkan naive N x N comparison?
2. Bagaimana ukuran dan distribusi block untuk setiap blocking key?
3. Strategi mana yang menghasilkan candidate set yang manageable?
4. Seberapa banyak candidate dari deterministic R1-R4 yang tercakup oleh setiap strategi blocking?

## Hypothesis

- Blocking berbasis kombinasi field seperti `city + dob` atau `surname prefix + dob` menghasilkan block yang lebih kecil daripada satu field umum.
- Blocking mengurangi computational cost, tetapi blocking yang terlalu ketat dapat membuang pasangan yang seharusnya dibandingkan.
- Coverage terhadap candidate deterministic dapat menjadi diagnostic internal, tetapi bukan recall karena ground truth entity belum tersedia.

## Scope and limitations

- Dataset raw tidak dibaca untuk matching dan tidak diubah.
- Missing component tidak pernah dipakai sebagai blocking key.
- `customer_id` tidak digunakan sebagai blocking key atau label evaluasi.
- Candidate coverage terhadap R1-R4 hanya reference coverage, bukan precision, recall, atau F1.
- Kandidat pair hanya dimaterialisasi jika jumlahnya berada di bawah batas memori eksperimen.

In [ ]:
from itertools import combinations
from pathlib import Path
from time import perf_counter
import tracemalloc

import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan. Jalankan 04_standardization.ipynb terlebih dahulu.')

df = pd.read_csv(DATA_PATH).reset_index(names='row_index')
required_columns = [
    'row_index', 'email_std', 'phone_digits_std', 'last_name_std',
    'dob_std', 'city_std', 'name_dob_key_std', 'email_phone_key_std',
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f'Kolom terstandardisasi belum tersedia: {missing_columns}')

print('File:', DATA_PATH)
print('Shape:', df.shape)

## Experiment 1 - Naive pairwise baseline

Baseline teoritis dihitung dengan $n(n-1)/2$. Tidak ada cross join yang dibuat. Angka ini dipakai untuk menghitung candidate reduction dari setiap blocking strategy.

In [ ]:
n_rows = len(df)
naive_pair_count = n_rows * (n_rows - 1) // 2
print(f'Records: {n_rows:,}')
print(f'Naive pair count: {naive_pair_count:,}')

## Experiment 2 - Define blocking keys

Setiap key hanya valid jika seluruh komponennya tersedia. Prefix digunakan untuk candidate generation, bukan sebagai bukti bahwa dua record adalah entity yang sama.

In [ ]:
def first_digits(series: pd.Series, length: int) -> pd.Series:
    values = series.astype('string')
    return values.where(values.str.len().ge(length)).str.slice(0, length)


def first_character(series: pd.Series) -> pd.Series:
    values = series.astype('string').str.strip()
    return values.where(values.str.len().gt(0)).str.slice(0, 1)


def combine_key(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    valid = frame[columns].notna().all(axis=1)
    key = pd.Series(pd.NA, index=frame.index, dtype='string')
    key.loc[valid] = frame.loc[valid, columns].astype('string').agg('|'.join, axis=1)
    return key

blocking_keys = {
    'email_domain': df['email_std'].astype('string').str.extract(r'@([^@]+)$', expand=False),
    'phone_prefix_7': first_digits(df['phone_digits_std'], 7),
    'dob': df['dob_std'].astype('string'),
    'surname_prefix_dob': combine_key(
        pd.DataFrame({
            'surname_prefix': first_character(df['last_name_std']),
            'dob_std': df['dob_std'],
        }),
        ['surname_prefix', 'dob_std'],
    ),
    'city_dob': combine_key(df, ['city_std', 'dob_std']),
}

blocking_key_frame = pd.DataFrame(blocking_keys)
blocking_key_frame.head()

## Experiment 3 - Candidate count, block distribution, runtime, and memory

Candidate pair dihitung dari ukuran setiap repeated block. Fungsi ini tidak membuat dataframe pair sehingga strategi dengan block besar tetap dapat diprofilkan secara aman.

In [ ]:
def profile_blocking_key(key_series: pd.Series, strategy: str) -> dict:
    start_time = perf_counter()
    tracemalloc.start()

    valid_values = key_series.dropna()
    block_sizes = valid_values.value_counts()
    repeated_sizes = block_sizes[block_sizes > 1]
    candidate_pairs = int((repeated_sizes * (repeated_sizes - 1) // 2).sum())
    eligible_rows = int(valid_values.size)
    largest_block = int(block_sizes.max()) if len(block_sizes) else 0

    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    runtime_seconds = perf_counter() - start_time

    return {
        'blocking_strategy': strategy,
        'eligible_rows': eligible_rows,
        'unique_blocks': int(block_sizes.size),
        'repeated_blocks': int(repeated_sizes.size),
        'rows_in_repeated_blocks': int(repeated_sizes.sum()),
        'candidate_pairs': candidate_pairs,
        'largest_block': largest_block,
        'candidate_reduction_percentage': (
            (1 - candidate_pairs / naive_pair_count) * 100
            if naive_pair_count else 0
        ),
        'runtime_seconds': runtime_seconds,
        'peak_memory_mb': peak_bytes / (1024 ** 2),
    }

blocking_summary = pd.DataFrame([
    profile_blocking_key(blocking_key_frame[strategy], strategy)
    for strategy in blocking_key_frame.columns
]).sort_values('candidate_pairs')

blocking_summary.round(3)

## Experiment 4 - Reference candidate coverage

R1-R4 dari deterministic matching dipakai sebagai reference candidate set. Coverage di sini hanya menjawab: "berapa banyak candidate deterministic yang juga ditemukan oleh blocking strategy?" Karena tidak ada ground truth, nilai ini bukan candidate recall dan bukan recall entity resolution.

In [ ]:
def build_candidate_pairs(key_series: pd.Series, strategy: str) -> set[tuple[int, int]]:
    pair_set = set()
    eligible = pd.DataFrame({'row_index': df['row_index'], 'block_key': key_series}).dropna(subset=['block_key'])
    for _, block in eligible.groupby('block_key', sort=False):
        row_indices = sorted(block['row_index'].astype(int).tolist())
        pair_set.update(combinations(row_indices, 2))
    return pair_set


def build_deterministic_reference() -> set[tuple[int, int]]:
    pair_set = set()
    reference_definitions = {
        'email_std': df['email_std'],
        'phone_digits_std': df['phone_digits_std'],
        'name_dob_key_std': df['name_dob_key_std'],
        'email_phone_key_std': df['email_phone_key_std'],
    }
    for key_series in reference_definitions.values():
        pair_set.update(build_candidate_pairs(key_series, 'deterministic_reference'))
    return pair_set


deterministic_reference_pairs = build_deterministic_reference()
reference_count = len(deterministic_reference_pairs)
coverage_rows = []
for row in blocking_summary.itertuples(index=False):
    if row.candidate_pairs > 2_000_000:
        coverage_rows.append({
            'blocking_strategy': row.blocking_strategy,
            'reference_candidate_pairs': reference_count,
            'blocking_candidate_pairs': row.candidate_pairs,
            'reference_pairs_covered': pd.NA,
            'reference_coverage_percentage': pd.NA,
            'coverage_status': 'not_materialized_above_pair_limit',
        })
        continue

    strategy_pairs = build_candidate_pairs(
        blocking_key_frame[row.blocking_strategy],
        row.blocking_strategy,
    )
    covered = len(deterministic_reference_pairs.intersection(strategy_pairs))
    coverage_rows.append({
        'blocking_strategy': row.blocking_strategy,
        'reference_candidate_pairs': reference_count,
        'blocking_candidate_pairs': row.candidate_pairs,
        'reference_pairs_covered': covered,
        'reference_coverage_percentage': covered / reference_count * 100 if reference_count else pd.NA,
        'coverage_status': 'materialized',
    })

coverage_summary = pd.DataFrame(coverage_rows)
coverage_summary.round(3)

## Experiment 5 - Persist manageable candidate pairs

Pair artifact hanya dibuat untuk strategy dengan paling banyak 2.000.000 pair. Output menyimpan `row_index`, nama strategy, dan block size; tidak menyimpan nilai customer.

In [ ]:
PAIR_LIMIT = 2_000_000
pair_rows = []
for row in blocking_summary.itertuples(index=False):
    if row.candidate_pairs > PAIR_LIMIT:
        continue
    key_series = blocking_key_frame[row.blocking_strategy]
    eligible = pd.DataFrame({'row_index': df['row_index'], 'block_key': key_series}).dropna(subset=['block_key'])
    block_sizes = eligible['block_key'].value_counts()
    for block_key, block in eligible.groupby('block_key', sort=False):
        row_indices = sorted(block['row_index'].astype(int).tolist())
        block_size = int(block_sizes[block_key])
        pair_rows.extend({
            'blocking_strategy': row.blocking_strategy,
            'left_row_index': left_index,
            'right_row_index': right_index,
            'block_size': block_size,
        } for left_index, right_index in combinations(row_indices, 2))

blocking_candidate_pairs = pd.DataFrame(pair_rows)
if blocking_candidate_pairs.empty:
    blocking_candidate_pairs = pd.DataFrame(columns=[
        'blocking_strategy', 'left_row_index', 'right_row_index', 'block_size'
    ])

print('Persistable candidate pairs:', len(blocking_candidate_pairs))
blocking_candidate_pairs.head()

In [ ]:
OUTPUT_DIR = DATA_PATH.parents[1] / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_OUTPUT_PATH = OUTPUT_DIR / 'blocking_candidate_summary.csv'
COVERAGE_OUTPUT_PATH = OUTPUT_DIR / 'blocking_reference_coverage.csv'
PAIRS_OUTPUT_PATH = OUTPUT_DIR / 'blocking_candidate_pairs.csv'

blocking_summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)
coverage_summary.to_csv(COVERAGE_OUTPUT_PATH, index=False)
blocking_candidate_pairs.to_csv(PAIRS_OUTPUT_PATH, index=False)

print('Saved:', SUMMARY_OUTPUT_PATH)
print('Saved:', COVERAGE_OUTPUT_PATH)
print('Saved:', PAIRS_OUTPUT_PATH)
print('Raw dataset still exists:', (DATA_PATH.parents[1] / 'raw' / 'crm_50000_customers_dirty_v3.csv').exists())

# Result, Analysis, and Decision

## Result aktual

Gunakan `blocking_summary`, `coverage_summary`, dan jumlah `blocking_candidate_pairs` sebagai sumber hasil aktual setelah notebook dijalankan. Jangan menyalin angka ke markdown secara manual karena hasil bergantung pada snapshot dataset dan environment.

## Analysis

- `candidate_reduction_percentage` mengukur pengurangan terhadap baseline teoritis N x N.
- `largest_block` dan distribusi repeated block membantu mengidentifikasi blocking key yang terlalu umum.
- `reference_coverage_percentage` bukan recall. Nilai tersebut hanya menunjukkan overlap terhadap candidate deterministic R1-R4.
- Strategy yang tidak dimaterialisasi karena melewati `PAIR_LIMIT` tetap valid untuk profiling, tetapi belum memiliki artifact pair atau coverage terhitung.
- Blocking yang menghasilkan candidate lebih sedikit belum tentu lebih baik jika membuang true match.

## Decision rule

Prioritaskan strategy yang memiliki candidate reduction tinggi, block size dapat dijelaskan, runtime dan memory manageable, serta reference coverage yang tidak terlalu rendah. Jangan memilih berdasarkan candidate count saja.

## Limitations

- Ground truth entity belum tersedia.
- Candidate coverage terhadap deterministic R1-R4 bukan precision, recall, atau F1.
- Runtime dan memory hanya berlaku untuk snapshot dan environment saat notebook dijalankan.
- Pair artifact dibatasi `PAIR_LIMIT = 2_000_000` untuk mencegah penggunaan memory yang tidak terkendali.

## Next Experiment

Jika blocking menghasilkan beberapa strategy yang layak, lanjutkan ke evaluasi manual atau `09_evaluation.ipynb` untuk membangun validation set. Jika belum ada label, tahap berikutnya yang paling aman adalah error analysis pada candidate pairs, bukan automatic merge.